In [1]:
pip install sympy


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Libraries

In [12]:
import numpy as np
from scipy import signal
import sympy as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

p = 1e-12;n = 1e-9;u = 1e-6;m = 1e-3;k = 1e3;M = 1e6;G = 1e9


In [13]:

from IPython.display import display, Math


def _round_expr_for_latex(expr, decimals=3):
    expr = sp.sympify(expr)
    repl = {}

    # Round all numeric atoms (Float/Rational/Integer) to fixed decimals
    for n in expr.atoms(sp.Number):
        if n.is_number:
            repl[n] = sp.Float(round(float(n), decimals))

    return expr.xreplace(repl)

def print_L(expr, label=None, decimals=3):
    expr = sp.sympify(expr)

    if decimals is not None:
        if expr.is_number:
            expr = sp.Float(round(float(expr), decimals))
        else:
            expr = _round_expr_for_latex(expr, decimals)
    tex = sp.latex(expr, full_prec=False)
    if label:
        display(Math(rf"{label} = {tex}"))
    else:
        display(Math(tex))


In [14]:
# ...existing code...
PREFIXES = [
    ("p", p), ("n", n), ("u", u), ("m", m),
    ("", 1.0), ("k", k), ("M", M), ("G", G),
]


# Auto-pick best prefix
AUTO_PREFIXES = [("G", G), ("M", M), ("k", k), ("", 1.0), ("m", m), ("u", u), ("n", n), ("p", p)]

def fmt_auto_unit(x, unit="", digits=3):
    x = float(x)
    ax = abs(x)

    if ax == 0:
        return f"{0:.{digits}f} {unit}".strip()

    for pre, scale in AUTO_PREFIXES:
        if ax >= scale:
            return f"{x/scale:.{digits}f} {pre}{unit}".strip()

    pre, scale = AUTO_PREFIXES[-1]
    return f"{x/scale:.{digits}f} {pre}{unit}".strip()
def arrx(array,unit):
    return [element*unit for element in array]

def print_u(x, unit="", digits=3, label=None):
    txt = fmt_auto_unit(x, unit, digits)
    print(f"{label}: {txt}" if label else txt)


# Main Code

In [16]:
f0 = 5e3 # 5 K Hz
L = 10e-3 # 10 mH
zeta = 0.2 # damping factor

In [24]:
wn = 2 * sp.pi * f0
C = sp.N(1 / (L * wn**2))
R = sp.N(2 * zeta* (L/C)**0.5)  # zeta = 0.2
print_u(wn, unit="rad/s")
print_u(C, unit="F")
print_u(R, unit="Ohm")

31.416 krad/s
101.321 nF
125.664 Ohm


In [32]:
Kp = (2.85e10)/(wn**2)-1
Ki = (4.5e14)/(wn**2)
Kd = (320e3-2*zeta*wn)/(wn**2)

print_u(Kp, label="Kp");print_u(Ki, label="Ki");print_u(Kd, label="Kd")


Kp: 27.877
Ki: 455.945 k
Kd: 311.495 u


In [34]:
Rp = 10e3 # 10 kOhm
Rfp = Kp * Rp
Ci = 1 / (Ki * Rp)
Cd = Kd / Rp
print_u(Rfp, unit="Ohm", label="Rfp")
print_u(Ci, unit="F", label="Ci")
print_u(Cd, unit="F", label="Cd")

Rfp: 278.765 kOhm
Ci: 219.325 pF
Cd: 31.150 nF
